# v8戦略 Stage 1: JSON/CSV基盤学習

## 段階的SFT（Sequential Format Learning）

本ノートブックは **v8戦略** の **Stage 1** です。

### Person Uの知見に基づく段階的SFT

Person Uが0.84を達成した核心的な発見：
- **フォーマットごとに段階的に学習** → 各フォーマットの限界点に到達可能
- **TOMLは他のフォーマットから学習している**（Person T発見）
- **ベースモデルは特定フォーマットに対して100%の文法正解率を持っている**

### 4段階の学習計画

```
Stage 1: JSON/CSV (800件) ← 本ノートブック
    ↓ マージしてHFにアップロード
Stage 2: YAML (500件) ← Stage 1モデルをベースに学習
    ↓ マージしてHFにアップロード
Stage 3: XML (500件) ← Stage 2モデルをベースに学習
    ↓ マージしてHFにアップロード
Stage 4: Mixed (1000件) ← Stage 3モデルをベースに最終調整
```

### Stage 1の目標
- **JSON: 100%**（パース成功率）
- **CSV: 100%**（パース成功率）
- このモデルをHugging Faceにマージしてアップロード

### データセット
- **v8_stage1_json_csv**: 800件（JSON 400件 + CSV 400件）
- 全サンプルがバリデーション通過済み
- 不要なプレフィックス/コードブロックなし

## Step 1: 依存関係インストール

In [ ]:
# ============================================================
# 0) 依存関係の固定（Colabの"環境ブレ"対策）
# ============================================================
!pip -q uninstall -y numpy pandas datasets trl transformers accelerate peft unsloth unsloth-zoo bitsandbytes xformers
!pip -q install "numpy==2.0.2" "pandas==2.2.2"
!pip -q install \
  "datasets==4.3.0" \
  "trl==0.24.0" \
  "transformers==4.56.2" \
  "accelerate==1.1.0" \
  "peft==0.13.2" \
  "bitsandbytes==0.45.0"
!pip -q install "unsloth-zoo==2025.12.7" "unsloth==2025.12.7"

In [ ]:
# ============================================================
# 0.1) バージョン確認
# ============================================================
import numpy as np, pandas as pd
import datasets, trl, transformers, torch
from unsloth import FastLanguageModel

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("datasets", datasets.__version__)
print("trl", trl.__version__)
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("unsloth import OK")

## Step 2: 環境変数設定（v8 Stage 1用）

In [ ]:
import os

# v8 Stage 1 設定
VERSION = "v8_stage1_json_csv"
TITLE_LINE = f"qwen3-4b-structured-output-lora-{VERSION}"

def _getenv(name: str, default: str):
    return os.environ.get(name, default)

def _getenv_int(name: str, default: int) -> int:
    try:
        return int(os.environ.get(name, str(default)))
    except Exception:
        return default

def _getenv_float(name: str, default: float) -> float:
    try:
        return float(os.environ.get(name, str(default)))
    except Exception:
        return default

# ============================================================
# v8 Stage 1 固有の設定
# ============================================================

# 1. モデル・データセット関連
# Stage 1: ベースモデルから開始
os.environ["SFT_BASE_MODEL"] = "Qwen/Qwen3-4B-Instruct-2507"
os.environ["SFT_DATASET_ID"] = "LOCAL:/content/train.json"  # v8_stage1_json_csv
os.environ["SFT_OUT_LORA_DIR"] = f"/content/lora_structeval_{VERSION}"

# 2. 学習の基本パラメータ
os.environ["SFT_SEED"] = "3407"
os.environ["SFT_VAL_RATIO"] = "0.05"
os.environ["SFT_MAX_SEQ_LEN"] = "1024"

# 3. LoRA (アダプタ) 設定
os.environ["SFT_LORA_R"] = "64"
os.environ["SFT_LORA_ALPHA"] = "128"
os.environ["SFT_LORA_DROPOUT"] = "0"
os.environ["SFT_LORA_TARGET_MODULES"] = "q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj"

# 4. ハイパーパラメータ
# Person Wの知見: 少ないデータ(800件)なのでepoch=2で十分
os.environ["SFT_EPOCHS"] = "2"
os.environ["SFT_PER_DEVICE_TRAIN_BS"] = "2"
os.environ["SFT_PER_DEVICE_EVAL_BS"] = "2"
os.environ["SFT_GRAD_ACCUM"] = "8"
os.environ["SFT_LR"] = "5e-5"  # 標準的な学習率から開始
os.environ["SFT_WARMUP_RATIO"] = "0.1"
os.environ["SFT_WEIGHT_DECAY"] = "0.05"

# 5. ステップ・保存設定
os.environ["SFT_MAX_STEPS"] = "-1"
os.environ["SFT_LOGGING_STEPS"] = "10"
os.environ["SFT_EVAL_STEPS"] = "50"
os.environ["SFT_SAVE_STEPS"] = "100"
os.environ["SFT_SAVE_TOTAL_LIMIT"] = "2"

# 6. 特殊学習設定
os.environ["SFT_MASK_COT"] = "1"
os.environ["SFT_OUTPUT_MARKERS"] = "Output:,OUTPUT:,Final:,Answer:,Result:,Response:"
os.environ["SFT_OUTPUT_LEARN_MODE"] = "after_marker"
os.environ["SFT_USE_UPSAMPLING"] = "0"
os.environ["SFT_UPSAMPLE_RULES"] = '{}'

print(f'=== v8 Stage 1: JSON/CSV基盤学習 ===')
print(f'  データセット: {VERSION} (800件、JSON 400件 + CSV 400件)')
print(f'  MAX_SEQ_LEN={os.environ["SFT_MAX_SEQ_LEN"]}, EPOCHS={os.environ["SFT_EPOCHS"]}, LR={os.environ["SFT_LR"]}')
print(f'  目標: JSON 100%, CSV 100%')

## Step 3: HuggingFace ログイン

In [ ]:
from google.colab import userdata
from huggingface_hub import login, HfApi

print("Attempting login...")
MY_TOKEN = userdata.get("HF_TOKEN")
try:
    if MY_TOKEN:
        clean_token = MY_TOKEN.strip()
        login(token=clean_token)
        api = HfApi()
        print("✅ Login successful using Colab Secret (HF_TOKEN).")
    else:
        raise ValueError("HF_TOKEN is empty in Secrets.")
except Exception as e:
    print(f"⚠️ Auto-login failed: {e}")
    print("Switching to interactive login...")
    login()
    api = HfApi()

## Step 4: WandB（Weights & Biases）ログイン

学習ログを可視化するためにWandBにログインします。

**事前準備:**
1. [wandb.ai](https://wandb.ai/) でアカウント作成
2. APIキーを取得: https://wandb.ai/authorize
3. Colabのシークレット（🔑アイコン）に `WANDB_API_KEY` として登録

In [ ]:
import wandb

WANDB_KEY = userdata.get("WANDB_API_KEY")
try:
    if WANDB_KEY:
        wandb.login(key=WANDB_KEY.strip())
        print("✅ WandB login successful using Colab Secret (WANDB_API_KEY).")
    else:
        raise ValueError("WANDB_API_KEY is empty in Secrets.")
except Exception as e:
    print(f"⚠️ Auto-login failed: {e}")
    print("Switching to interactive login...")
    wandb.login()

# WandB run初期化
wandb.init(
    project="structeval-sft",
    name=VERSION,
    config={
        "base_model": os.environ.get("SFT_BASE_MODEL"),
        "max_seq_len": os.environ.get("SFT_MAX_SEQ_LEN"),
        "epochs": os.environ.get("SFT_EPOCHS"),
        "lr": os.environ.get("SFT_LR"),
        "lora_r": os.environ.get("SFT_LORA_R"),
    },
)
print(f"✅ WandB run initialized: {wandb.run.name}")

## Step 5: データセットのアップロード

**手順:**
1. ローカルPCから `inputs/sft_processed/v8_stage1_json_csv/train.json` をダウンロード
2. 下のセルを実行してアップロード

**v8_stage1_json_csvの特徴:**
- サンプル数: 800件（JSON 400件 + CSV 400件）
- 全サンプルがバリデーション通過済み
- 不要なプレフィックス/コードブロックなし

In [ ]:
import json
import shutil
from google.colab import files

print("train.json (v8_stage1_json_csv) をアップロードしてください...")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.json'):
        target_path = '/content/train.json'
        if filename != 'train.json':
            shutil.move(filename, target_path)
        print(f"✅ アップロード完了: {target_path}")
        with open(target_path, 'r') as f:
            data = json.load(f)
        print(f"   データ件数: {len(data)} 件")
        
        # フォーマット分布確認
        from collections import Counter
        formats = []
        for sample in data:
            if 'metadata' in sample and isinstance(sample['metadata'], dict):
                fmt = sample['metadata'].get('format', 'unknown')
                formats.append(fmt)
        print(f"   フォーマット分布: {Counter(formats)}")
        break
else:
    print("⚠️ JSONファイルがアップロードされませんでした")

## Step 6: 学習の実行

**注意**: 以下のセルは標準SFTコードとほぼ同じです。
主な変更点は環境変数（Step 2で設定済み）のみ。

In [ ]:
import random
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple
from datasets import load_dataset, Dataset
from transformers import TrainingArguments, Trainer, TrainerCallback

# 環境変数を再読み込み
BASE_MODEL_ID = _getenv("SFT_BASE_MODEL", "Qwen/Qwen3-4B-Instruct-2507")
DATASET_ID    = _getenv("SFT_DATASET_ID", "u-10bei/structured_data_with_cot_dataset_512_v2")
OUT_LORA_DIR  = _getenv("SFT_OUT_LORA_DIR", "/content/lora_structeval_t_qwen3_4b")
SEED        = _getenv_int("SFT_SEED", 3407)
VAL_RATIO   = _getenv_float("SFT_VAL_RATIO", 0.05)
MAX_SEQ_LEN = _getenv_int("SFT_MAX_SEQ_LEN", 1024)
LORA_R       = _getenv_int("SFT_LORA_R", 64)
LORA_ALPHA   = _getenv_int("SFT_LORA_ALPHA", 128)
LORA_DROPOUT = _getenv_float("SFT_LORA_DROPOUT", 0)
LORA_TARGET_MODULES = _getenv("SFT_LORA_TARGET_MODULES", "q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj").split(",")
NUM_TRAIN_EPOCHS            = _getenv_int("SFT_EPOCHS", 1)
PER_DEVICE_TRAIN_BATCH_SIZE = _getenv_int("SFT_PER_DEVICE_TRAIN_BS", 2)
PER_DEVICE_EVAL_BATCH_SIZE  = _getenv_int("SFT_PER_DEVICE_EVAL_BS", 2)
GRAD_ACCUM                  = _getenv_int("SFT_GRAD_ACCUM", 8)
LR                          = _getenv_float("SFT_LR", 5e-5)
WARMUP_RATIO                = _getenv_float("SFT_WARMUP_RATIO", 0.1)
MAX_STEPS        = _getenv_int("SFT_MAX_STEPS", -1)
LOGGING_STEPS    = _getenv_int("SFT_LOGGING_STEPS", 10)
EVAL_STEPS       = _getenv_int("SFT_EVAL_STEPS", 50)
SAVE_STEPS       = _getenv_int("SFT_SAVE_STEPS", 100)
SAVE_TOTAL_LIMIT = _getenv_int("SFT_SAVE_TOTAL_LIMIT", 2)
WEIGHT_DECAY     = _getenv_float("SFT_WEIGHT_DECAY", 0.05)

print(f"BASE_MODEL_ID: {BASE_MODEL_ID}")
print(f"DATASET_ID: {DATASET_ID}")
print(f"OUT_LORA_DIR: {OUT_LORA_DIR}")

In [ ]:
# ============================================================
# データセット読み込み
# ============================================================
def load_sft_dataset(dataset_id: str):
    """SFTデータセットを読み込む"""
    if dataset_id.startswith("LOCAL:"):
        local_path = dataset_id.replace("LOCAL:", "")
        with open(local_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return Dataset.from_list(data)
    else:
        return load_dataset(dataset_id, split="train")

raw_ds = load_sft_dataset(DATASET_ID)
print(f"Loaded {len(raw_ds)} samples")

# messagesカラムの存在確認
if "messages" not in raw_ds.column_names:
    raise ValueError("Dataset must have 'messages' column")

# train/val分割
random.seed(SEED)
indices = list(range(len(raw_ds)))
random.shuffle(indices)
val_size = int(len(raw_ds) * VAL_RATIO)
val_indices = indices[:val_size]
train_indices = indices[val_size:]

train_ds = raw_ds.select(train_indices)
val_ds = raw_ds.select(val_indices)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

In [ ]:
# ============================================================
# モデルとトークナイザーのロード
# ============================================================
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,  # 自動検出
    load_in_4bit=True,
)

# LoRAアダプタの追加
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print("Model loaded with LoRA adapter")

In [ ]:
# ============================================================
# データの前処理（トークン化）
# ============================================================
import re

# CoTマスク設定
MASK_COT = _getenv("SFT_MASK_COT", "1") in ("1", "true", "True")
OUTPUT_MARKERS = _getenv("SFT_OUTPUT_MARKERS", "Output:,OUTPUT:,Final:,Answer:,Result:,Response:").split(",")
OUTPUT_LEARN_MODE = _getenv("SFT_OUTPUT_LEARN_MODE", "after_marker")

def find_output_marker_position(text: str) -> int:
    """Output:などのマーカー位置を見つける"""
    for marker in OUTPUT_MARKERS:
        pos = text.find(marker)
        if pos != -1:
            if OUTPUT_LEARN_MODE == "after_marker":
                return pos + len(marker)
            else:
                return pos
    return -1

def preprocess_function(examples):
    """メッセージをトークン化する"""
    texts = []
    for messages in examples["messages"]:
        # chat_templateを適用
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    
    # トークン化
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
        return_tensors=None,
    )
    
    # labelsの設定（assistant部分のみ学習対象）
    tokenized["labels"] = tokenized["input_ids"].copy()
    
    return tokenized

# データセットのトークン化
tokenized_train = train_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=train_ds.column_names,
    desc="Tokenizing train"
)

tokenized_val = val_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=val_ds.column_names,
    desc="Tokenizing val"
)

print(f"Tokenized train: {len(tokenized_train)}, val: {len(tokenized_val)}")

In [ ]:
# ============================================================
# Trainerの設定と学習実行
# ============================================================
from transformers import DataCollatorForSeq2Seq

# データコレーター
# 可変長のSFT学習では DataCollatorForSeq2Seq が推奨されます。
# バッチごとに最長系列に合わせて動的にパディングを行います。
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
)

# TrainingArguments
training_args = TrainingArguments(
    output_dir=OUT_LORA_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    max_steps=MAX_STEPS,
    seed=SEED,
    bf16=True,  # L4/A100用
    fp16=False,
    optim="adamw_8bit",
    report_to="wandb",
    run_name=VERSION,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

# 学習実行
print("Starting training...")
trainer.train()

# モデル保存
trainer.save_model(OUT_LORA_DIR)
tokenizer.save_pretrained(OUT_LORA_DIR)
print(f"✅ Model saved to {OUT_LORA_DIR}")

# WandBセッション終了
if wandb.run is not None:
    wandb.finish()
    print("✅ WandBセッションを終了しました")

## Step 7: マージモデルをHugging Faceにアップロード

**重要**: 段階的SFTでは、このモデルを次のStageのベースモデルとして使います。

Person Hのコードに基づき、LoRAをベースモデルにマージしてアップロードします。

In [ ]:
# ============================================================
# マージモデルのアップロード（Stage間継承用）
# ============================================================
from pathlib import Path
import torch
from peft import PeftModel

# 設定
LORA_SAVE_DIR = Path(OUT_LORA_DIR)
HF_REPO_ID = _getenv("HF_REPO_ID", f"kmd2525/{VERSION}-merged")
PRIVATE = _getenv("HF_PRIVATE", "1") in ("1", "true", "True")

print(f"Merging LoRA adapter to base model...")
print(f"  LORA_SAVE_DIR: {LORA_SAVE_DIR}")
print(f"  HF_REPO_ID: {HF_REPO_ID}")

# 一度GPUメモリを解放
del model
del trainer
torch.cuda.empty_cache()

# ベースモデルを通常ロード（4bitではなく）
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=torch.bfloat16,
    load_in_4bit=False,
)

# LoRAアダプタをロード
print(f"Loading LoRA adapter from: {LORA_SAVE_DIR}")
model = PeftModel.from_pretrained(base_model, str(LORA_SAVE_DIR))

# マージ
print("Merging LoRA into base (merge_and_unload)...")
model = model.merge_and_unload()

# 一時保存
STAGE_DIR = Path("/content/hf_upload_stage_merged")
if STAGE_DIR.exists():
    shutil.rmtree(STAGE_DIR)
STAGE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Saving merged model to: {STAGE_DIR}")
model.save_pretrained(str(STAGE_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(STAGE_DIR))

print(f"📦 Upload contents: {sorted([p.name for p in STAGE_DIR.iterdir()])}")

In [ ]:
# ============================================================
# HuggingFaceにアップロード
# ============================================================
# ★ HF_REPO_IDを実際のリポジトリ名に変更してから実行

# Model Card (README.md) を作成
model_card_content = f'''---
license: apache-2.0
base_model: {BASE_MODEL_ID}
tags:
  - structured-output
  - json
  - csv
  - sft
  - sequential-format-learning
language:
  - en
  - ja
---

# {VERSION}-merged

## Model Description

This model is **Stage 1** of the Sequential Format Learning (v8 strategy) for structured data output.

### Training Strategy

Based on Person U\'s approach that achieved 0.84 on the leaderboard:
- Train one format at a time
- Merge LoRA to base model after each stage
- Use merged model as the base for the next stage

### Stage 1 Focus: JSON & CSV

- **Formats**: JSON (400 samples) + CSV (400 samples) = 800 total
- **Goal**: 100% parse success rate for JSON and CSV
- **Base Model**: `{BASE_MODEL_ID}`

### Training Parameters

- MAX_SEQ_LEN: {MAX_SEQ_LEN}
- EPOCHS: {NUM_TRAIN_EPOCHS}
- Learning Rate: {LR}
- LoRA R: {LORA_R}, Alpha: {LORA_ALPHA}

### Sequential Format Learning Pipeline

```
Stage 1: JSON/CSV (800) ← This model
    ↓
Stage 2: YAML (500)
    ↓
Stage 3: XML (500)
    ↓
Stage 4: Mixed/TOML (1000)
    ↓
Final Model → LB 0.8+
```

### Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("{HF_REPO_ID}")
tokenizer = AutoTokenizer.from_pretrained("{HF_REPO_ID}")
```

### Next Stage

Use this model as the base for Stage 2 (YAML training):
```python
os.environ["SFT_BASE_MODEL"] = "{HF_REPO_ID}"
```
'''

# README.md を保存
readme_path = STAGE_DIR / "README.md"
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(model_card_content)
print(f"✅ Model Card saved: {readme_path}")

api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="model",
    exist_ok=True,
    private=PRIVATE,
)

api.upload_folder(
    folder_path=str(STAGE_DIR),
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message=f"Upload merged SFT model ({VERSION})",
)

print("✅ アップロード完了!")
print(f"URL: https://huggingface.co/{HF_REPO_ID}")
print("")
print("=== 次のステップ ===")
print(f"Stage 2では、ベースモデルを以下に変更してください:")
print(f'  os.environ["SFT_BASE_MODEL"] = "{HF_REPO_ID}"')

## ✅ Stage 1 完了

### 次のステップ: Stage 2（YAML強化）

1. `v8_stage2_yaml.ipynb` を開く
2. ベースモデルを以下に変更:
   ```python
   os.environ["SFT_BASE_MODEL"] = "kmd2525/v8_stage1_json_csv-merged"
   ```
3. データセット: `inputs/sft_processed/v8_stage2_yaml/train.json` をアップロード
4. 学習を実行

### 参考: 段階的SFTの全体フロー

```
Stage 1: JSON/CSV (800件) → JSON 100%, CSV 100%
    ↓ マージしてHFにアップロード
Stage 2: YAML (500件) → YAML 100%
    ↓ マージしてHFにアップロード
Stage 3: XML (500件) → XML 95%+
    ↓ マージしてHFにアップロード
Stage 4: Mixed (1000件) → TOML 90%+
    ↓
最終モデル → LB 0.8+
```

## Step 8: Colabインスタンスの削除（課金停止）

⚠️ **重要**: 学習・アップロード完了後、不要な課金を防ぐためColabインスタンスを削除してください。

以下のセルを実行すると、ランタイムが終了しインスタンスが削除されます。

In [ ]:
# ============================================================
# Colabインスタンスの削除（課金停止）
# ============================================================
# 以下を実行するとランタイムが終了し、インスタンスが削除されます
# 必ずHFへのアップロードが完了してから実行してください！

from google.colab import runtime
runtime.unassign()